# Fine-mapping methods with a 500-dimensional sketch

In [ ]:
[finemap_DEN_SLA]
parameter: genofile = paths
parameter: cwd = path("output")
parameter: ld_dir = path            
parameter: B_stoc = 500
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}_DEN_SLALOM.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R); library(tidyverse); library(MASS); library(crayon)
    library(pecotmr); library(susieR); library(vroom); library(pgenlibr); library(Rfast)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/", full.names = TRUE, pattern = "\\.R$")) source(f)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/", full.names = TRUE, pattern = "\\.R$")) source(f)

    compute_LD_from_sketch <- function(U, p) {
      U <- as.matrix(U)
      if (ncol(U) == p) Uv <- U else if (nrow(U) == p) Uv <- t(U) else
        stop(sprintf("Sketch U (%d x %d) matches neither dim to p = %d", nrow(U), ncol(U), p))
      R <- cor(Uv); R[is.na(R)] <- 0; R
    }

    ## ---- load pheno + matched B=500 LD -------------------------------------
    path    <- ${_input:r}
    dat     <- readRDS(path)
    n         <- dat$n
    n_snps    <- dat$p
    true_vars <- dat$true_vars

    prefix  <- str_remove(basename(path), "_ncausal_.*$")
    ld_path <- file.path("${ld_dir}", paste0(prefix, "_ukb_B_stoc_", ${B_stoc}, ".rds"))
    stopifnot(file.exists(ld_path))
    ld_dat  <- readRDS(ld_path)

    sumstat <- dat$sumstat_qc     # must carry z (+ chrom/pos/variant_id/A1/A2 for RAISS)

    refs <- list(
      insample = list(sumstat = sumstat, LD = ld_dat$LD_full),
      subset   = list(sumstat = sumstat, LD = ld_dat$LD_sample),
      sketch   = list(sumstat = sumstat, U  = ld_dat$U)
    )

    ## ---- helpers (unchanged) ----------------------------------------------
    run_raiss_on_outliers <- function(outlier_flags, z_vec, ref_df, LD_mat) {
      known_idx <- which(!outlier_flags)
      known_df <- data.frame(chrom = ref_df$chrom[known_idx], pos = ref_df$pos[known_idx],
        variant_id = ref_df$variant_id[known_idx], A1 = ref_df$A1[known_idx],
        A2 = ref_df$A2[known_idx], z = z_vec[known_idx], stringsAsFactors = FALSE)
      result <- raiss(ref_panel = ref_df, known_zscores = known_df, LD_matrix = LD_mat,
                      lamb = 0.01, rcond = 0.01, R2_threshold = 0.6, minimum_ld = 5, verbose = FALSE)
      combined_z <- z_vec
      if (!is.null(result) && !is.null(result$result_nofilter)) {
        raiss_df <- result$result_nofilter
        for (i in which(outlier_flags)) {
          vid <- ref_df$variant_id[i]; ri <- which(raiss_df$variant_id == vid)
          if (length(ri) == 1 && !is.na(raiss_df$z[ri])) combined_z[i] <- raiss_df$z[ri]
        }
      }
      list(z = combined_z, raiss_result = result)
    }

    finemap_QC <- function(input, n, name) {
      z <- input$sumstat$z; p <- length(z)
      LD <- if (!is.null(input$LD)) input$LD
            else if (!is.null(input$U)) compute_LD_from_sketch(input$U, p)
            else stop("input must contain 'LD' or 'U'")
      ref_df <- input$sumstat %>%
      tidyr::separate(SNP, into = c("chrom", "pos", "A2", "A1"),
                      sep = ":", remove = FALSE, convert = TRUE) %>%
      dplyr::rename(variant_id = SNP)

      dentist_sw <- dentistSingleWindow(zScore = z, R = LD, nSample = n)
      dentist_outlier_flags <- dentist_sw$outlier
      slalom_result <- slalom(zScore = z, R = LD)
      slalom_outliers <- slalom_result$data$outliers; slalom_outliers[is.na(slalom_outliers)] <- FALSE

      slalom_raiss  <- run_raiss_on_outliers(slalom_outliers, z, ref_df, LD)
      dentist_raiss <- run_raiss_on_outliers(dentist_outlier_flags, z, ref_df, LD)

      dentist_qc_z <- z; dentist_qc_z[dentist_outlier_flags] <- dentist_sw$imputed_z[dentist_outlier_flags]
      slalom_raiss_z  <- slalom_raiss$z
      dentist_raiss_z <- dentist_raiss$z

      run_susie <- function(z_vec, LD, n)
        susie_rss(z = z_vec, R = LD, n = n, control = susie_rss_control(check_prior = FALSE))

      dentist_qc_res    <- run_susie(dentist_qc_z, LD, n);    dentist_qc_res$outlier_number    <- sum(dentist_outlier_flags)
      dentist_raiss_res <- run_susie(dentist_raiss_z, LD, n); dentist_raiss_res$outlier_number <- sum(dentist_outlier_flags)
      slalom_raiss_res  <- run_susie(slalom_raiss_z, LD, n);  slalom_raiss_res$outlier_number  <- sum(slalom_outliers)

      list(dentist_qc_res = dentist_qc_res, dentist_raiss_res = dentist_raiss_res,
           slalom_raiss = slalom_raiss_res, input_path = path)
    }

    result_list <- lapply(names(refs), function(nm) finemap_QC(refs[[nm]], n = n, name = nm))
    names(result_list) <- names(refs)

    output <- list(result_list = result_list, n_sample = n, input_path = path,
                   true_vars = true_vars, n_snp = n_snps, B_stoc = ${B_stoc})
    saveRDS(output, ${_output:r})

In [ ]:
[finemap_CARMA]
parameter: genofile = paths
parameter: cwd = path("output")
parameter: ld_dir = path
parameter: B_stoc = 500
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R); library(tidyverse); library(MASS); library(crayon)
    library(pecotmr); library(susieR); library(vroom); library(pgenlibr); library(Rfast); library(CARMA)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/", full.names = TRUE, pattern = "\\.R$")) source(f)

    compute_LD_from_sketch <- function(U, p) {
      U <- as.matrix(U)
      if (ncol(U) == p) Uv <- U else if (nrow(U) == p) Uv <- t(U) else
        stop(sprintf("Sketch U (%d x %d) matches neither dim to p = %d", nrow(U), ncol(U), p))
      R <- cor(Uv); R[is.na(R)] <- 0; R
    }

    path    <- ${_input:r}
    dat     <- readRDS(path)
    n         <- dat$n
    true_vars <- dat$true_vars

    prefix  <- str_remove(basename(path), "_ncausal_.*$")
    ld_path <- file.path("${ld_dir}", paste0(prefix, "_ukb_B_stoc_", ${B_stoc}, ".rds"))
    stopifnot(file.exists(ld_path))
    ld_dat  <- readRDS(ld_path)

    sumstat <- dat$sumstat_qc
    refs <- list(
      insample = list(sumstat = sumstat, LD = ld_dat$LD_full),
      subset   = list(sumstat = sumstat, LD = ld_dat$LD_sample),
      sketch   = list(sumstat = sumstat, U  = ld_dat$U)
    )

    finemap_CARMA <- function(input, n, name) {
      z <- input$sumstat$z; p <- length(z)
      LD <- if (!is.null(input$LD)) input$LD
            else if (!is.null(input$U)) compute_LD_from_sketch(input$U, p)
            else stop("input must contain 'LD' or 'U'")
      CARMA.results <- CARMA(list(z), list(as.matrix(LD)), lambda.list = list(1), outlier.switch = TRUE)
      CARMA.results$sets <- list()
      real_set <- if (length(CARMA.results[[1]]$`Credible set`[[2]]) != 0)
                    CARMA.results[[1]]$`Credible set`[[2]] else NULL
      CARMA.results$sets$cs <- real_set
      CARMA.results
    }

    result_list <- lapply(names(refs), function(nm) finemap_CARMA(refs[[nm]], n = n, name = nm))
    names(result_list) <- names(refs)

    output <- list(result_list = result_list, n_sample = n, input_path = path,
                   true_vars = true_vars, n_snp = dat$p, B_stoc = ${B_stoc})
    saveRDS(output, ${_output:r})

In [ ]:
[finemap_rsparse_pro]
parameter: genofile = paths
parameter: cwd = path("output")
parameter: ld_dir = path
parameter: B_stoc = 500
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}.rsparsepro.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)

    compute_LD_from_sketch <- function(U, p) {
      U <- as.matrix(U)
      if (ncol(U) == p) Uv <- U else if (nrow(U) == p) Uv <- t(U) else
        stop(sprintf("Sketch U (%d x %d) matches neither dim to p = %d", nrow(U), ncol(U), p))
      R <- cor(Uv); R[is.na(R)] <- 0; R
    }

    path    <- ${_input:r}
    dat     <- readRDS(path)
    true_vars <- dat$true_vars

    prefix  <- str_remove(basename(path), "_ncausal_.*$")
    ld_path <- file.path("${ld_dir}", paste0(prefix, "_ukb_B_stoc_", ${B_stoc}, ".rds"))
    stopifnot(file.exists(ld_path))
    ld_dat  <- readRDS(ld_path)

    sumstat <- dat$sumstat_qc
    refs <- list(
      insample = list(sumstat = sumstat, LD = ld_dat$LD_full),
      subset   = list(sumstat = sumstat, LD = ld_dat$LD_sample),
      sketch   = list(sumstat = sumstat, U  = ld_dat$U)
    )

    finemap_Rsparse <- function(input, out_prefix,
                                python_bin = "python",
                                rsparsepro_script = "/sc/arion/projects/ad-omics/haochen/rss_qc/RSparsePro_LD/src/rsparsepro_ld.py") {
      z <- input$sumstat$z; p <- length(z)
      LD <- if (!is.null(input$LD)) input$LD
            else if (!is.null(input$U)) compute_LD_from_sketch(input$U, p)
            else stop("input must contain 'LD' or 'U'")
      LD <- as.matrix(LD)
      rsid <- colnames(LD); if (is.null(rsid)) rsid <- paste0("snp_", seq_along(z))

      z_df <- tibble(RSID = rsid, Z = z)
      z_file <- paste0(out_prefix, ".z.txt"); ld_file <- paste0(out_prefix, ".ld.txt")
      result_path <- paste0(out_prefix, ".rsparsepro.txt")

      write_tsv(z_df, z_file)
      write.table(LD, file = ld_file, sep = " ", row.names = FALSE, col.names = FALSE, quote = FALSE)

      cmd <- paste(shQuote(python_bin), shQuote(rsparsepro_script),
                   "--z", shQuote(z_file), "--ld", shQuote(ld_file), "--save", shQuote(out_prefix))
      system(cmd)

      res <- read_tsv(result_path, show_col_types = FALSE) %>% mutate(index = row_number())
      if (!"cs" %in% names(res)) stop("RSparsePro output missing column 'cs'.")
      cs_index <- setdiff(unique(res$cs), 0)

      result_rds <- list(fit = res, set = list())
      result_rds$set$cs <- if (length(cs_index) == 0) NULL else
        lapply(cs_index, function(k) res %>% filter(cs == k) %>% pull(index))
      list(fit = result_rds)
    }

    out_base <- ${_output:r}
    result_list <- lapply(names(refs), function(nm)
      finemap_Rsparse(refs[[nm]], out_prefix = paste0(out_base, "_", nm)))   # distinct scratch per ref
    names(result_list) <- names(refs)

    output <- list(result_list = result_list, n_sample = dat$n, input_path = path,
                   true_vars = true_vars, n_snp = dat$p, B_stoc = ${B_stoc})
    saveRDS(output, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKBB_B500_DEN_SLALOM"
job_dir="${base_dir}/${job_name}"
data_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result"
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/06.Fig2C.Finite_Reference_Other_Methods.ipynb"

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"
work_dir="${job_dir}"

cat << EOF > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:\$PATH:/usr/bin:/bin

sos run "${nb}" finemap_DEN_SLA \\
    --genofile \$(echo ${data_dir}/h2g_H2G/causal_CAUSAL/pheno/*.rds) \\
    --ld_dir ${data_dir}/B_500/ld --B_stoc 500 \\
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh
base_sh=${job_name}.sh

for causal in 1 2 3; do
  for h2g in 0.0004 0.0006 0.0008; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
        sb "${output_script}"
  done
done

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKBB_B500_CARMA"
job_dir="${base_dir}/${job_name}"
data_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result"
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/06.Fig2C.Finite_Reference_Other_Methods.ipynb"

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"
work_dir="${job_dir}"

cat << EOF > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:\$PATH:/usr/bin:/bin

sos run "${nb}" finemap_CARMA \\
    --genofile \$(echo ${data_dir}/h2g_H2G/causal_CAUSAL/pheno/*.rds) \\
    --ld_dir ${data_dir}/B_500/ld --B_stoc 500 \\
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh
base_sh=${job_name}.sh

for causal in 1 2 3; do
  for h2g in 0.0004 0.0006 0.0008; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
        sb "${output_script}"
  done
done

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="UKBB_B500_rsparse_pro"
job_dir="${base_dir}/${job_name}"
data_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_sketch_dimension/result"
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/06.Fig2C.Finite_Reference_Other_Methods.ipynb"

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"
work_dir="${job_dir}"

cat << EOF > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:\$PATH:/usr/bin:/bin

sos run "${nb}" finemap_rsparse_pro \\
    --genofile \$(echo ${data_dir}/h2g_H2G/causal_CAUSAL/pheno/*.rds) \\
    --ld_dir ${data_dir}/B_500/ld --B_stoc 500 \\
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh
base_sh=${job_name}.sh

for causal in 1 2 3; do
  for h2g in 0.0004 0.0006 0.0008; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
        sb "${output_script}"
  done
done

In [ ]:
[DEN_SLA_summary]
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}_METHOD.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)

    `%||%` <- function(a, b) if (is.null(a) || length(a) == 0 || all(is.na(a))) b else a

    ## ---- helpers -------------------------------------------------------------
    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      length(intersect(variant, unlist(cs)))
    }
    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      sum(vapply(cs, function(s) length(intersect(variant, s)) > 0, logical(1)))
    }
    parse_field <- function(f, key) {
      as.numeric(str_match(f, paste0(key, "_(-?[0-9.]+)"))[, 2])
    }

    ## ---- config --------------------------------------------------------------
    path  <- ${folder:r}
    files <- list.files(path, pattern = "region_.*\\.rds$",
                        recursive = TRUE, full.names = TRUE)
    stopifnot(length(files) > 0)

    ## ---- detect which method scheme is present -------------------------------
    known_schemes <- list(
      ld_ref    = c("insample", "subset", "sketch"),
      flip      = c("gtex_original", "gtex_flip"),
      reference = c("ADSP", "gtex", "kg", "UKB")
    )
    available     <- names(readRDS(files[[1]])[["result_list"]])
    scheme        <- Filter(function(s) all(s %in% available), known_schemes)
    susie_methods <- if (length(scheme) >= 1) scheme[[1]] else available
    message("Using susie_methods: ", paste(susie_methods, collapse = ", "))

    # QC pipelines: label -> where its credible set / outlier count live in the fit
    qc_specs <- list(
      DENTIST_QC    = list(cs    = function(s) s$dentist_qc_res$sets$cs,
                           n_out = function(s) s$dentist_qc_res$outlier_number),
      DENTIST_RAISS = list(cs    = function(s) s$dentist_raiss_res$sets$cs,
                           n_out = function(s) s$dentist_raiss_res$outlier_number),
      SLALOM_RAISS  = list(cs    = function(s) s$slalom_raiss$sets$cs,
                           n_out = function(s) s$slalom_raiss$outlier_number)
    )

    ## ---- build one long table, reading each file only once -------------------
    summary_tb <- map_dfr(files, function(f) {
      res     <- readRDS(f)
      variant <- res$true_vars
      meta <- tibble(
        input_file = f,
        B_stoc     = res$B_stoc %||% parse_field(f, "B_stoc"),   # B dimension
        ncausal    = parse_field(f, "ncausal"),
        miss_prop  = parse_field(f, "miss_prop"),
        miss_fac   = parse_field(f, "miss_fac")
      )
      map_dfr(susie_methods, function(m) {
        sus <- res[["result_list"]][[m]]
        if (is.null(sus)) return(NULL)
        map_dfr(names(qc_specs), function(qc) {
          cs    <- qc_specs[[qc]]$cs(sus)
          n_out <- tryCatch(qc_specs[[qc]]$n_out(sus), error = function(e) NULL)
          bind_cols(meta, tibble(
            method           = m,
            qc               = qc,
            outlier_number   = n_out %||% NA_real_,
            true_var_number  = discover_var_num(variant, cs),
            true_set_number  = true_set_num(variant, cs),
            total_var_number = length(variant),
            total_set_number = length(cs),
            total_set_length = length(unlist(cs))
          ))
        })
      })
    })

    ## ---- write one file per QC pipeline (same names as before) ---------------
    out_path <- ${_output:r}
    walk(names(qc_specs), function(qc_label) {
      saveRDS(filter(summary_tb, qc == qc_label),
              str_replace(out_path, "METHOD", qc_label))
    })

    # combined long table
    saveRDS(summary_tb, str_replace(out_path, "METHOD", "ALL"))

In [ ]:
[single_QC_summary]
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)

    `%||%` <- function(a, b) if (is.null(a) || length(a) == 0 || all(is.na(a))) b else a

    ## ---- helpers -------------------------------------------------------------
    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      length(intersect(variant, unlist(cs)))
    }
    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      sum(vapply(cs, function(s) length(intersect(variant, s)) > 0, logical(1)))
    }
    parse_field <- function(f, key) {
      as.numeric(str_match(f, paste0(key, "_(-?[0-9.]+)"))[, 2])
    }
    # CS location differs by method: CARMA $sets$cs ; plain $set$cs ; RSparsePro $fit$set$cs
    get_cs <- function(sus) {
      if (is.null(sus))             return(NULL)
      if (!is.null(sus$sets$cs))    return(sus$sets$cs)      # CARMA
      if (!is.null(sus$set$cs))     return(sus$set$cs)       # plain set$cs
      if (!is.null(sus$fit$set$cs)) return(sus$fit$set$cs)   # RSparsePro
      NULL
    }

    ## ---- config --------------------------------------------------------------
    path  <- ${folder:r}
    files <- list.files(path, pattern = "region_.*\\.rds$",
                        recursive = TRUE, full.names = TRUE)
    stopifnot(length(files) > 0)

    # method scheme present in result_list; ld_ref = the LD-reference conditions
    known_schemes <- list(
      ld_ref    = c("insample", "subset", "sketch"),
      flip      = c("gtex_original", "gtex_flip"),
      reference = c("ADSP", "gtex", "kg", "UKB")
    )
    available     <- names(readRDS(files[[1]])[["result_list"]])
    scheme        <- Filter(function(s) all(s %in% available), known_schemes)
    susie_methods <- if (length(scheme) >= 1) scheme[[1]] else available
    message("Using susie_methods: ", paste(susie_methods, collapse = ", "))

    ## ---- build summary (each file read once) ---------------------------------
    summary_tb <- map_dfr(files, function(f) {
      res     <- readRDS(f)
      variant <- res$true_vars
      meta <- tibble(
        input_file = f,
        B_stoc     = res$B_stoc %||% parse_field(f, "B_stoc"),   # B dimension
        ncausal    = parse_field(f, "ncausal"),
        miss_prop  = parse_field(f, "miss_prop"),
        miss_fac   = parse_field(f, "miss_fac")
      )
      map_dfr(susie_methods, function(m) {
        sus <- res[["result_list"]][[m]]
        if (is.null(sus)) return(NULL)
        cs <- get_cs(sus)
        bind_cols(meta, tibble(
          method           = m,
          true_var_number  = discover_var_num(variant, cs),
          true_set_number  = true_set_num(variant, cs),
          total_var_number = length(variant),
          total_set_number = length(cs),
          total_set_length = length(unlist(cs))
        ))
      })
    })

    saveRDS(summary_tb, ${_output:r})

## Summary

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B500_DEN_SLALOM/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/06.Fig2C.Finite_Reference_Other_Methods.ipynb" DEN_SLA_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B500_CARMA/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/06.Fig2C.Finite_Reference_Other_Methods.ipynb" single_QC_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B500_rsparse_pro/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/06.Fig2C.Finite_Reference_Other_Methods.ipynb" single_QC_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [1]:
res = readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B500_rsparse_pro/result/h2g_0.0004/causal_1/region_320_ncausal_1_h2g_0.0004.rsparsepro.rds")

In [2]:
res$result_list$sketch$fit$set

$cs
$cs[[1]]
[1] 757